In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import numpy as np
from unet import *
import datasets
import training
import os, re
from skimage.metrics import structural_similarity

# Prevents crashes when showing graphs
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [6]:
train_sims = np.load("../train_sims.npy")
train_sims = train_sims[train_sims < 750]
val_sims = np.load("../val_sims.npy")
val_sims = val_sims[val_sims < 750]
test_sims = np.load("../test_sims.npy")
test_sims = test_sims[test_sims < 750]


In [ ]:
dataset_arguments = {
    "points_per_side": 3,
    "radius": 5,
    "steps": (0, 200),
    "types": [0],          # binary only (or [0,1] for mixed)
    "channels": "all",
    "future_delta": 0
}

print(train_sims.min(), train_sims.max(), train_sims.shape)
print(val_sims.min(), val_sims.max(), val_sims.shape)

model, train_losses, val_losses = training.train_from_scratch(
    dataset_type=datasets.FixedThinDatasetFull,   # fast training
    dataset_arguments=dataset_arguments,
    train_sims=train_sims,
    val_sims=val_sims,
    optimizer=lambda params: torch.optim.Adam(params, lr=1e-3),
    min_epochs=5,
    max_epochs=20,
    device=device,
    loss_weights_init=(1.0, 0.0),                 # baseline: MSE only
    physics_fn=training.physics_none              # or training.darcy_loss
)

1 749 (470,)
2 747 (114,)


  0%|          | 0/20 [00:00<?, ?it/s]